# Parquet + Pandas

用2026主流方案（Parquet + pandas/pyarrow）实现：

- 写入时间序列数据
- 按“天”分区存储
- 按时间范围快速读取

👉 对标 tstables 的 append + read_range

## 1. 生成示例数据

In [1]:
import numpy as np
import pandas as pd

def generate_sample_data(rows=10_000, cols=5, seed=42):
    """Generate a reproducible financial-like DataFrame."""
    rng = np.random.default_rng(seed)
    index = pd.date_range("2021-01-01", periods=rows, freq="min")

    steps = rng.normal(loc=0.0, scale=0.02, size=(rows, cols))
    data = 100 + np.cumsum(steps, axis=0)
    columns = [f"No{i}" for i in range(cols)]
    return pd.DataFrame(data, index=index, columns=columns).round(4)

data = generate_sample_data(rows=10_000, cols=5)
data.head()

,No0,No1,No2,No3,No4
2021-01-01 00:00:00,100.0061,99.9792,100.0150,100.0188,99.9610
2021-01-01 00:01:00,99.9801,99.9818,100.0087,100.0185,99.9439
2021-01-01 00:02:00,99.9976,99.9973,100.0100,100.0410,99.9533
2021-01-01 00:03:00,99.9805,100.0047,99.9908,100.0586,99.9523
2021-01-01 00:04:00,99.9768,99.9911,100.0153,100.0555,99.9437


## 2. 写入（分区 + append）

In [2]:
# !pip install -U pyarrow pandas

In [3]:
from pathlib import Path

def write_partitioned(df, base_path="dataset"):
    base = Path(base_path)
    base.mkdir(exist_ok=True)

    df = df.reset_index().rename(columns={"index": "timestamp"})
    df["date"] = df["timestamp"].dt.date

    df.to_parquet(
        base,
        engine="pyarrow",
        partition_cols=["date"],
        index=False
    )

write_partitioned(data)
print("write done")

write done


## 3. 追加写入（模拟流式数据）

In [5]:
# 模拟新增数据（下一天）
new_data = generate_sample_data(rows=50_000)
new_data.index = new_data.index + pd.Timedelta(days=2)

write_partitioned(new_data)
print("append done")

append done


## 4. 按时间范围读取（核心能力）

In [6]:
start = "2021-01-02"
end = "2021-01-03"

df = pd.read_parquet(
    "dataset",
    engine="pyarrow",
    filters=[
        ("date", ">=", start),
        ("date", "<=", end)
    ]
)

df.head()

,timestamp,No0,No1,No2,No3,No4,date
0,2021-01-02 00:00:00,99.6694,100.3856,99.7412,100.2496,99.6347,2021-01-02
1,2021-01-02 00:01:00,99.6531,100.4114,99.7565,100.2357,99.6355,2021-01-02
2,2021-01-02 00:02:00,99.6471,100.3888,99.7600,100.2274,99.6165,2021-01-02
3,2021-01-02 00:03:00,99.6176,100.3575,99.7526,100.2257,99.5990,2021-01-02
4,2021-01-02 00:04:00,99.6306,100.3562,99.7641,100.2220,99.5832,2021-01-02


In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4320 entries, 0 to 4319
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   timestamp  4320 non-null   datetime64[us]
 1   No0        4320 non-null   float64       
 2   No1        4320 non-null   float64       
 3   No2        4320 non-null   float64       
 4   No3        4320 non-null   float64       
 5   No4        4320 non-null   float64       
 6   date       4320 non-null   category      
dtypes: category(1), datetime64[us](1), float64(5)
memory usage: 207.5 KB


## 5. Compaction

In [8]:
# --- Compaction 说明 ---
#
# 为什么会有小文件问题：
# - 数据通常是多次 append / 并行写入的
# - 每次写入都会生成新的 parquet 文件
# - 长期下来，每个分区（如 date=xxx）下面会积累大量小文件
#
# 为什么小文件不好：
# - 读取时需要打开大量文件，metadata + IO 开销大
# - 无法高效顺序读取，性能明显下降
# - 文件系统（尤其是对象存储）处理大量文件效率较低
#
# compaction 如何解决小文件问题：
# - 读取整个数据集（或某些分区）
# - 合并成更大的数据块
# - 按原有 partition 重新写回
# - 多个小文件 → 少量大文件（读取性能显著提升）
#
# 使用方式：
# - 定期运行 compaction（而不是每次写入后都做）
# - 常见策略：按天 / 按周 / 数据量达到阈值时触发
# - 写入阶段多文件保证速度，读取前合并文件保证性能
#
# ------------------------------------

In [10]:
import pyarrow.dataset as ds
import pyarrow.parquet as pq

def compact_dataset(src="dataset", dst="dataset_compacted"):
    dataset = ds.dataset(src, format="parquet", partitioning="hive")

    table = dataset.to_table()  # 合并读取（生产环境可分批）

    pq.write_to_dataset(
        table,
        root_path=dst,
        partition_cols=["date"]
    )

compact_dataset()
print("compaction done")

compaction done
